# الانحدار الخطي المتعدد - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات واستكشافها
3. ترميز المتغير الفئوي
4. فحص القيم الشاذة والعلاقات
5. الارتباط والتعدد الخطي (VIF)
6. تجهيز X و y
7. تقسيم البيانات والقياس
8. تدريب النموذج والتنبؤ
9. تفسير المعاملات والفحوصات الإحصائية
10. تحليل البواقي والتقييم


In [ ]:
# الخطوة 1) استيراد المكتبات الأساسية
# numpy: حسابات رقمية
# matplotlib / seaborn: رسوم وتحليل بصري
# pandas: تحميل ومعالجة البيانات الجدولية
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd


In [ ]:
# الخطوة 2) قراءة البيانات
# إذا كنت تعمل محليًا غيّر المسار إلى مكان الملف عندك (مثال: 50_Startups.csv)
dataset = pd.read_csv('/content/50_Startups.csv')

# عرض أول صفوف لفهم شكل البيانات
dataset.head()


In [ ]:
# الخطوة 3) استكشاف بصري أولي قبل الترميز
# الهدف: معرفة العلاقة العامة بين كل ميزة وبين الربح Profit
features = dataset.columns[:-1]  # جميع الأعمدة ما عدا العمود الأخير (Profit)
print("الميزات الموجودة:", features.tolist())

for feature in features:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=dataset, x=feature, y='Profit')
    plt.title(f'العلاقة بين {feature} و Profit')
    plt.xlabel(feature)
    plt.ylabel('Profit')
    plt.show()


In [ ]:
# الخطوة 4) ترميز المتغير الفئوي State
# One-Hot Encoding يحول الولايات إلى أعمدة رقمية
# drop_first=True لتجنب فخ المتغيرات الوهمية (Dummy Variable Trap)
dataset_encoded = pd.get_dummies(dataset, columns=['State'], drop_first=True, dtype=int)
dataset_encoded.head()


In [ ]:
# الخطوة 5) فحص القيم الشاذة بطريقة IQR + Boxplot
# الفكرة: أي قيمة خارج النطاق [Q1 - 1.5*IQR, Q3 + 1.5*IQR] تعتبر شاذة
for col in dataset_encoded.select_dtypes(include=np.number).columns:
    Q1 = dataset_encoded[col].quantile(0.25)
    Q3 = dataset_encoded[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = dataset_encoded[(dataset_encoded[col] < lower_bound) | (dataset_encoded[col] > upper_bound)]
    print(f"\nالقيم الشاذة في {col}:")
    print(outliers)

    plt.figure(figsize=(7, 4))
    sns.boxplot(x=dataset_encoded[col])
    plt.title(f'Boxplot - {col}')
    plt.show()


In [ ]:
# الخطوة 6) مقارنة كل ميزة مع Profit بعد الترميز
features = dataset_encoded.columns.tolist()
features.remove('Profit')

plt.figure(figsize=(12, 8))
for i, feature in enumerate(features):
    plt.subplot((len(features) + 1) // 2, 2, i + 1)
    plt.scatter(dataset_encoded[feature], dataset_encoded['Profit'], alpha=0.7)
    plt.xlabel(feature)
    plt.ylabel('Profit')
    plt.title(f'{feature} vs Profit')

plt.tight_layout()
plt.show()


In [ ]:
# الخطوة 7) ترتيب الأعمدة لتوحيد الشكل قبل بناء النموذج
column_order = [
    'R&D Spend',
    'Administration',
    'Marketing Spend',
    'State_Florida',
    'State_New York',
    'Profit'
]
dataset_encoded = dataset_encoded[column_order]
dataset_encoded.head()


In [ ]:
# الخطوة 8) مصفوفة الارتباط
# مهمة لشرح: أي الميزات لها ارتباط أكبر/أقل مع Profit
correlation_matrix = dataset_encoded.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('مصفوفة الارتباط')
plt.show()


In [ ]:
# الخطوة 9) فحص التعدد الخطي باستخدام VIF
# VIF عالي جدًا يعني وجود ترابط قوي بين الميزات وقد يضعف التفسير
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X_vif = dataset_encoded.drop(columns=['Profit']).astype(float)
X_vif['constant'] = 1

vif_data = pd.DataFrame()
vif_data['Feature'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
print(vif_data)


In [ ]:
# الخطوة 10) تجهيز مصفوفة الخصائص X والمتغير الهدف y
X = dataset_encoded.iloc[:, :-1].values
y = dataset_encoded.iloc[:, -1].values

print('شكل X:', X.shape)
print('شكل y:', y.shape)


In [ ]:
# الخطوة 11) تقسيم البيانات تدريب/اختبار
# 80% تدريب و 20% اختبار
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


In [ ]:
# الخطوة 12) قياس الخصائص Feature Scaling
# مفيد عندما تختلف المقاييس بين الأعمدة بشكل كبير
from sklearn.preprocessing import StandardScaler
sc_X = StandardScaler()
X_train = sc_X.fit_transform(X_train)
X_test = sc_X.transform(X_test)


In [ ]:
# الخطوة 13) تدريب نموذج الانحدار الخطي المتعدد
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()
regressor.fit(X_train, y_train)


In [ ]:
# الخطوة 14) التنبؤ على بيانات الاختبار
y_pred = regressor.predict(X_test)

print('أول 5 قيم حقيقية:', y_test[:5])
print('أول 5 قيم متوقعة:', y_pred[:5])


In [ ]:
# الخطوة 15) التفسير الإحصائي (OLS Summary)
# نعرض ملخصًا يتضمن p-values، R-squared وغيرها
X_train_with_constant = sm.add_constant(X_train)
model = sm.OLS(y_train, X_train_with_constant)
results = model.fit()
print(results.summary())


In [ ]:
# الخطوة 16) استخراج المعاملات من نموذج sklearn
# كل معامل يوضح مقدار تغير الربح عند زيادة ميزة واحدة (مع ثبات الباقي)
coefficients = regressor.coef_
intercept = regressor.intercept_

feature_names = dataset_encoded.columns[:-1].tolist()
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})
print(coef_df)
print('Intercept:', intercept)


In [ ]:
# الخطوة 17) تقييم النموذج بمقاييس أساسية
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f'MAE  = {mae:,.2f}')
print(f'MSE  = {mse:,.2f}')
print(f'RMSE = {rmse:,.2f}')
print(f'R2   = {r2:.4f}')


In [ ]:
# الخطوة 18) تحليل البواقي (Residual Diagnostics)
# إذا توزعت البواقي حول الصفر بشكل عشوائي فهذه إشارة جيدة غالبًا
residuals = y_test - y_pred

plt.figure(figsize=(7, 5))
plt.scatter(y_pred, residuals, color='blue')
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('الربح المتوقع')
plt.ylabel('البواقي (الخطأ)')
plt.title('البواقي مقابل القيم المتوقعة')
plt.show()


## ملاحظات للمحاضر
- إذا كان `R2` منخفضًا، ناقش مع الطلاب أسباب ذلك (حجم البيانات، جودة الميزات، ضجيج البيانات).
- إذا ظهرت VIF عالية، وضّح معنى التعدد الخطي وتأثيره على التفسير.
- يمكنك مقارنة هذا النموذج مع نماذج أخرى لاحقًا (Ridge/Lasso) لشرح معالجة multicollinearity.
